# 01 - Data Exploration

Explore the AeroConform data pipeline: configuration, synthetic trajectory
generation, delta encoding, robust normalization, windowing, and visualization.

This notebook demonstrates the full preprocessing flow that converts raw
ADS-B state vectors into model-ready training samples.

## Configuration

All hyperparameters and paths live in `AeroConformConfig`, powered by
pydantic-settings. Values can be overridden via environment variables
prefixed with `AEROCONFORM_`.

In [ ]:
from aeroconform.config import AeroConformConfig

config = AeroConformConfig()

print("Target FIR:", config.target_fir)
print("Bounding box:", config.bbox)
print("Features:", config.features)
print("Sequence length:", config.seq_len)
print("Patch length:", config.patch_len)
print("Number of patches:", config.num_patches)
print("Input dim:", config.input_dim)
print("Window stride:", config.window_stride)
print("\nModel config:")
print(f"  d_model={config.d_model}, n_heads={config.n_heads}, n_layers={config.n_layers}")
print(f"  d_ff={config.d_ff}, n_components={config.n_components}")
print(f"  output_dim={config.output_dim}")

## Generate Synthetic Trajectories

We generate realistic synthetic ADS-B trajectories over the LIMM FIR
(northern Italy) for demonstration. Each trajectory contains 6 features:
latitude, longitude, barometric altitude, velocity, true track, and vertical rate.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)


def generate_synthetic_trajectory(
    n_steps: int = 200,
    rng: np.random.Generator = rng,
) -> np.ndarray:
    """Generate a single realistic synthetic ADS-B trajectory."""
    lat_start = rng.uniform(config.bbox[0], config.bbox[1])
    lon_start = rng.uniform(config.bbox[2], config.bbox[3])
    alt = rng.uniform(15000, 40000)
    vel = rng.uniform(200, 500)
    hdg = rng.uniform(0, 360)
    vrate = 0.0

    traj = np.zeros((n_steps, 6))
    for t in range(n_steps):
        traj[t] = [lat_start, lon_start, alt, vel, hdg, vrate]
        # Update with small perturbations
        lat_start += vel * np.cos(np.radians(hdg)) / 3600 / 60
        lon_start += vel * np.sin(np.radians(hdg)) / 3600 / 60 / np.cos(np.radians(lat_start))
        alt += vrate / 60
        vel += rng.normal(0, 1)
        vel = np.clip(vel, 150, 600)
        hdg += rng.normal(0, 0.3)
        hdg = hdg % 360
        vrate = rng.normal(0, 50)
    return traj


# Generate 20 synthetic trajectories
trajectories = [generate_synthetic_trajectory(n_steps=200, rng=rng) for _ in range(20)]
print(f"Generated {len(trajectories)} trajectories")
print(f"Shape of each: {trajectories[0].shape}")
print(f"Features: {config.features}")
print(f"\nSample row (first trajectory, first timestep):")
print(f"  lat={trajectories[0][0, 0]:.4f}, lon={trajectories[0][0, 1]:.4f}, "
      f"alt={trajectories[0][0, 2]:.0f}ft, vel={trajectories[0][0, 3]:.1f}kts, "
      f"hdg={trajectories[0][0, 4]:.1f}deg, vrate={trajectories[0][0, 5]:.1f}fpm")

## Visualize Raw Trajectories

Plot the raw latitude/longitude tracks over the LIMM bounding box.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))

for i, traj in enumerate(trajectories):
    ax.plot(traj[:, 1], traj[:, 0], alpha=0.6, linewidth=0.8)

# Draw bounding box
bbox = config.bbox
ax.plot(
    [bbox[2], bbox[3], bbox[3], bbox[2], bbox[2]],
    [bbox[0], bbox[0], bbox[1], bbox[1], bbox[0]],
    "r--", linewidth=2, label="LIMM FIR bbox",
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Synthetic ADS-B Trajectories over LIMM FIR")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Delta Encoding

The foundation model operates on delta-encoded state vectors: the difference
between consecutive timesteps. Heading (true_track) uses shortest angular
difference to handle the 360/0 wrap-around.

In [ ]:
from aeroconform.data.preprocessing import delta_encode

traj = trajectories[0]
deltas = delta_encode(traj)

print(f"Original shape: {traj.shape}")
print(f"Delta shape:    {deltas.shape}  (T-1 because of differencing)")
print(f"\nFirst 5 delta rows:")
print(deltas[:5])

# Plot delta distributions
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
feature_names = ["d_lat", "d_lon", "d_alt", "d_vel", "d_hdg", "d_vrate"]

for idx, (ax, name) in enumerate(zip(axes.flat, feature_names)):
    all_deltas = np.concatenate([delta_encode(t)[:, idx] for t in trajectories])
    ax.hist(all_deltas, bins=50, alpha=0.7, color="steelblue", edgecolor="white")
    ax.set_title(name)
    ax.set_xlabel("Delta value")
    ax.set_ylabel("Count")
    ax.axvline(0, color="red", linestyle="--", alpha=0.5)

plt.suptitle("Delta-Encoded Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

## Robust Normalization

ADS-B data contains outliers, so we use median/IQR normalization instead
of mean/std: `x_norm = (x - median) / (IQR + eps)`.

In [ ]:
from aeroconform.data.preprocessing import compute_norm_stats, normalize, denormalize

# Delta-encode all trajectories
delta_trajs = [delta_encode(t) for t in trajectories]

# Compute normalization statistics
stats = compute_norm_stats(delta_trajs)
print("Normalization statistics:")
print(f"  Median: {[f'{m:.6f}' for m in stats['median']]}")
print(f"  IQR:    {[f'{q:.6f}' for q in stats['iqr']]}")

# Normalize
normed = normalize(delta_trajs[0], stats)
print(f"\nNormalized range per feature:")
for i, name in enumerate(feature_names):
    all_normed = np.concatenate([normalize(d, stats)[:, i] for d in delta_trajs])
    print(f"  {name}: [{all_normed.min():.2f}, {all_normed.max():.2f}], "
          f"median={np.median(all_normed):.4f}")

# Verify round-trip
reconstructed = denormalize(normed, stats)
max_error = np.abs(reconstructed - delta_trajs[0]).max()
print(f"\nRound-trip max error: {max_error:.2e}")

## Windowing

Trajectories are sliced into fixed-length windows of `seq_len=128` timesteps
with 50% overlap (stride = seq_len / 2). Shorter windows are zero-padded
with a boolean mask.

In [ ]:
from aeroconform.data.preprocessing import window_trajectory

# Window a single normalized delta trajectory
normed_traj = normalize(delta_encode(trajectories[0]), stats)
windows = window_trajectory(normed_traj, icao24="abc123", start_time=0, config=config)

print(f"Trajectory length: {normed_traj.shape[0]}")
print(f"seq_len={config.seq_len}, stride={config.window_stride}")
print(f"Number of windows: {len(windows)}")

for i, w in enumerate(windows):
    valid_frac = w['mask'].sum() / len(w['mask'])
    print(f"  Window {i}: shape={w['data'].shape}, "
          f"valid={valid_frac:.1%}, icao24={w['icao24']}")

## Full Preprocessing Pipeline

The `preprocess_pipeline` function runs the full chain: extract trajectories
from a DataFrame, delta-encode, normalize, and window.

In [ ]:
import pandas as pd
from aeroconform.data.preprocessing import preprocess_pipeline

# Build a DataFrame mimicking OpenSky state vectors
rows = []
for i, traj in enumerate(trajectories):
    icao24 = f"ac{i:04d}"
    for t in range(len(traj)):
        rows.append({
            "icao24": icao24,
            "timestamp": t,
            "latitude": traj[t, 0],
            "longitude": traj[t, 1],
            "baro_altitude": traj[t, 2],
            "velocity": traj[t, 3],
            "true_track": traj[t, 4],
            "vertical_rate": traj[t, 5],
            "on_ground": False,
        })

states_df = pd.DataFrame(rows)
print(f"States DataFrame: {states_df.shape}")
print(states_df.head())

# Run full pipeline
all_windows, norm_stats = preprocess_pipeline(states_df, config)
print(f"\nTotal windows produced: {len(all_windows)}")
print(f"Each window data shape: {all_windows[0]['data'].shape}")

## PyTorch Dataset

The `TrajectoryDataset` wraps windowed trajectories into PyTorch samples
with input, target (shifted patches), and mask tensors.

In [ ]:
from aeroconform.data.dataset import TrajectoryDataset

dataset = TrajectoryDataset(all_windows, config=config)
print(f"Dataset size: {len(dataset)}")

sample = dataset[0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"  input shape:  {sample['input'].shape}  (seq_len, input_dim)")
print(f"  target shape: {sample['target'].shape}  (num_patches, patch_len*input_dim)")
print(f"  mask shape:   {sample['mask'].shape}  (seq_len,)")
print(f"  metadata:     {sample['metadata']}")

## Visualize Processed Samples

Plot the delta-encoded, normalized features from a training sample,
along with the validity mask.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10))

sample_data = sample["input"].numpy()
sample_mask = sample["mask"].numpy()

for idx, (ax, name) in enumerate(zip(axes.flat, feature_names)):
    ax.plot(sample_data[:, idx], color="steelblue", linewidth=0.8)
    # Shade invalid regions
    invalid = ~sample_mask
    if invalid.any():
        ax.fill_between(
            range(len(invalid)), ax.get_ylim()[0], ax.get_ylim()[1],
            where=invalid, alpha=0.2, color="red", label="Padded",
        )
    ax.set_title(f"Normalized {name}")
    ax.set_xlabel("Timestep")
    ax.grid(True, alpha=0.3)

plt.suptitle("Processed Training Sample (Delta-Encoded, Normalized)", fontsize=14)
plt.tight_layout()
plt.show()

## Synthetic Anomalies Preview

AeroConform includes an `AnomalyInjector` that generates realistic attack
scenarios. Let us preview one GPS spoofing injection.

In [ ]:
from aeroconform.data.synthetic_anomalies import AnomalyInjector

injector = AnomalyInjector(rng=np.random.default_rng(42))
clean_traj = trajectories[0]
spoofed, labels = injector.inject_gps_spoofing(clean_traj, offset_nm=5.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Spatial plot
ax = axes[0]
ax.plot(clean_traj[:, 1], clean_traj[:, 0], "b-", label="Clean", alpha=0.7)
ax.plot(spoofed[:, 1], spoofed[:, 0], "r-", label="GPS Spoofed", alpha=0.7)
onset = np.argmax(labels)
ax.plot(spoofed[onset, 1], spoofed[onset, 0], "ro", markersize=10, label="Spoof onset")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("GPS Spoofing: Spatial View")
ax.legend()
ax.grid(True, alpha=0.3)

# Labels over time
ax = axes[1]
ax.fill_between(range(len(labels)), 0, labels, alpha=0.3, color="red", label="Anomalous")
ax.set_xlabel("Timestep")
ax.set_ylabel("Label")
ax.set_title("GPS Spoofing: Anomaly Labels")
ax.set_ylim(-0.1, 1.1)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated the AeroConform data pipeline:

1. **Configuration** via `AeroConformConfig` (pydantic-settings)
2. **Synthetic trajectory generation** with realistic ADS-B features
3. **Delta encoding** with heading wrap-around handling
4. **Robust normalization** using median/IQR
5. **Windowing** with overlap and padding
6. **Full preprocessing pipeline** from DataFrame to PyTorch dataset
7. **Anomaly injection** preview (GPS spoofing)

Next: `02_pretraining.ipynb` trains the trajectory foundation model.